# R9 — Late-Delivery Compensation Engine

Deterministic compensation policy layer (SOP-grounded) over the policy-free gold table. Builds `hop.genai.compensation_decisions`: one row per order with fault, is_late, eligible, compensation_type, amount and reason_code. Validated against the 24 compensation probe tickets in `hop.genai.compensation_probes`.

In [0]:
# Config cell — change model names here only, if a built-in model name changes
CATALOG = "hop"
CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"
EMBED_MODEL = "databricks-gte-large-en"
AS_OF_DATE = "2026-08-31"
print(CATALOG, CHAT_MODEL, EMBED_MODEL, AS_OF_DATE)

## Policy (from the SOP docs — binding, order matters)

**The four facts** (SOP 01): fault, delivered?, late beyond its class threshold?, tier.

**Fault from issue code** (SOP 01): `KITCHEN_DELAY`/`WRONG_ITEM` → restaurant;
`RIDER_DELAY`/`DAMAGED_IN_TRANSIT` → rider; `CUSTOMER_UNAVAILABLE` → customer;
`OTHER` or no ticket → unknown. Data guarantee: at most one ticket per order (verified).

**Late** (SOP 02/03/04): delivered AND minutes past promise **> 5** (instant) / **> 20**
(standard), measured from the `accepted` event (gold's `minutes_past_promise` already is).
Cancelled orders are never late.

**Decision precedence:**
1. customer at fault → **NONE** (no coupon, no refund — SOP 05)
2. cancelled (not customer fault) → **REFUND 100% of order value** (SOP 06; tier ignored)
3. late → **COUPON**: base Rs 40 (instant) / Rs 60 (standard); × tier multiplier
   (free ×1, hop_plus ×1.5, hop_pro ×2) **only when restaurant or rider at fault**;
   unknown fault → base alone (SOP 03/04/05)
4. otherwise → **NONE** (on time)

In [0]:
# R9 — build hop.genai.compensation_decisions (one row per order, 13,152 expected)
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.genai.compensation_decisions AS
WITH base AS (
  SELECT g.order_id, g.status, g.delivery_class, g.order_value, g.customer_tier,
         g.minutes_past_promise, t.issue_code
  FROM {CATALOG}.gold.orders_enriched g
  LEFT JOIN {CATALOG}.raw.support_tickets t ON g.order_id = t.order_id
),
facts AS (
  SELECT *,
    CASE issue_code
      WHEN 'KITCHEN_DELAY' THEN 'restaurant'
      WHEN 'WRONG_ITEM'    THEN 'restaurant'
      WHEN 'RIDER_DELAY'   THEN 'rider'
      WHEN 'DAMAGED_IN_TRANSIT' THEN 'rider'
      WHEN 'CUSTOMER_UNAVAILABLE' THEN 'customer'
      ELSE 'unknown'              -- OTHER or no ticket
    END AS fault,
    (status = 'delivered'
      AND ((delivery_class = 'instant'  AND minutes_past_promise > 5)
        OR (delivery_class = 'standard' AND minutes_past_promise > 20))) AS is_late
  FROM base
)
SELECT
  order_id, issue_code, fault, customer_tier,
  status, delivery_class, minutes_past_promise, is_late,
  CASE WHEN fault = 'customer'  THEN false
       WHEN status = 'cancelled' THEN true
       WHEN is_late              THEN true
       ELSE false END AS eligible,
  CASE WHEN fault = 'customer'  THEN 'NONE'
       WHEN status = 'cancelled' THEN 'REFUND'
       WHEN is_late              THEN 'COUPON'
       ELSE 'NONE' END AS compensation_type,
  CASE WHEN fault = 'customer'  THEN 0.0
       WHEN status = 'cancelled' THEN order_value
       WHEN is_late THEN
         CASE delivery_class WHEN 'instant' THEN 40.0 ELSE 60.0 END
         * CASE WHEN fault IN ('restaurant','rider')
                THEN CASE customer_tier WHEN 'hop_pro' THEN 2.0
                                        WHEN 'hop_plus' THEN 1.5
                                        ELSE 1.0 END
                ELSE 1.0 END
       ELSE 0.0 END AS amount,
  CASE WHEN fault = 'customer'  THEN 'CUSTOMER_FAULT'
       WHEN status = 'cancelled' THEN 'NOT_DELIVERED'
       WHEN is_late AND fault = 'restaurant' THEN 'LATE_RESTAURANT_FAULT'
       WHEN is_late AND fault = 'rider'      THEN 'LATE_RIDER_FAULT'
       WHEN is_late AND fault = 'unknown'     THEN 'LATE_UNKNOWN_FAULT'
       ELSE 'ON_TIME' END AS reason_code
FROM facts
""")
dec = spark.table(f"{CATALOG}.genai.compensation_decisions")
print("decisions:", dec.count())
display(dec.groupBy("compensation_type", "reason_code").count().orderBy("compensation_type", "reason_code"))

In [0]:
# Verification 1 — integrity: one row per gold order, decision invariants
from pyspark.sql import functions as F

gold = spark.table(f"{CATALOG}.gold.orders_enriched").select("order_id", "order_value")
chk = (dec.count(), dec.select("order_id").distinct().count(), gold.count())
print("decision rows / distinct orders / gold orders:", chk)

inv = dec.filter(
    ((F.col("eligible") & (F.col("compensation_type") == 'NONE')) |
     (~F.col("eligible") & (F.col("compensation_type") != 'NONE')) |
     (F.col("amount") < 0))
).count()
print("invariant violations (eligible<->type, negative amount):", inv)

refund_bad = (dec.join(gold, "order_id")
    .filter(F.col("compensation_type") == 'REFUND')
    .filter(F.abs(F.col("amount") - F.col("order_value")) > 1e-9).count())
print("REFUND rows whose amount != order_value:", refund_bad)

late_bad = dec.filter("is_late AND NOT (status = 'delivered')").count()
print("is_late rows that are not delivered:", late_bad)

In [0]:
# Verification 2 — independent amount recomputation (different code path)
base_map = F.create_map(F.lit("instant"), F.lit(40.0), F.lit("standard"), F.lit(60.0))
mult_map = F.create_map(F.lit("free"), F.lit(1.0), F.lit("hop_plus"), F.lit(1.5),
                        F.lit("hop_pro"), F.lit(2.0))

recheck = dec.join(gold, "order_id").select(
    "order_id", "amount",
    F.when(F.col("fault") == 'customer', F.lit(0.0))
     .when(F.col("status") == 'cancelled', F.col("order_value"))
     .when(F.col("is_late"),
           base_map[F.col("delivery_class")]
           * F.when(F.col("fault").isin("restaurant", "rider"),
                    mult_map[F.col("customer_tier")]).otherwise(F.lit(1.0)))
     .otherwise(F.lit(0.0)).alias("amount_recomputed")
)
mismatches = recheck.filter(F.abs(F.col("amount") - F.col("amount_recomputed")) > 1e-9).count()
print("amount mismatches (independent recompute):", mismatches, "of", recheck.count())

In [0]:
# Verification 3 — the 24 compensation probe tickets
probes = (spark.table(f"{CATALOG}.genai.compensation_probes")
    .join(spark.table(f"{CATALOG}.raw.support_tickets"), "ticket_id")
    .join(dec.drop("issue_code"), "order_id", "left"))

resolved = probes.filter(F.col("compensation_type").isNotNull()).count()
unresolved = probes.filter(F.col("compensation_type").isNull()).count()
print(f"probes resolved: {resolved} / 24 (unresolved = order not in gold: {unresolved})")
print("unresolved probe orders:", [r.order_id for r in probes.filter(F.col('compensation_type').isNull()).select('order_id').collect()])

display(probes.select(
    "ticket_id", "order_id", "issue_code", "fault", "customer_tier",
    "status", "delivery_class", F.round("minutes_past_promise", 2).alias("min_past_promise"),
    "is_late", "compensation_type", F.round("amount", 2).alias("amount"), "reason_code"
).orderBy("ticket_id"))